# 🏋️ Exercise 3 — Load
**Tujuan:** memuat data bersih ke PostgreSQL di schema pribadi Anda dan memverifikasinya.

> Isi bagian bertanda `# TODO`, lalu jalankan sel **Cek Jawaban**.

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [1]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

Konfigurasi OK — login database sebagai peserta25 @ ep-wandering-math-b3bml8w1-pooler.c-4.ap-southeast-1.aws.neon.tech


In [2]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

In [3]:
engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME),
                       pool_size=1, max_overflow=1, pool_pre_ping=True)   # hemat koneksi: ±70 peserta memakai server yang sama
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_schema()")).fetchone())

('peserta25', 'peserta25')


## Data bersih (sudah disiapkan)

In [4]:
def extract_sales(api_url, page_size=100):
    """Mengambil seluruh halaman data dari API."""
    rows, page = [], 1
    while True:
        resp = requests.get(page_url(api_url, page, page_size), timeout=30)
        resp.raise_for_status()                      # error jika status bukan 2xx
        payload = resp.json()
        rows.extend(payload["data"])
        print(f"Halaman {page}: {len(payload['data'])} baris")
        if not payload.get("has_next"):
            break
        page += 1
    return pd.DataFrame(rows)


def load_csv_fallback(filename):
    """Cadangan jika API gagal: baca CSV dari folder data/ (clone lokal) atau langsung dari GitHub."""
    for p in (filename, f"data/{filename}", f"../data/{filename}"):
        if os.path.exists(p):
            return pd.read_csv(p, dtype=str, keep_default_na=False)
    url = f"{RAW_BASE}/data/{filename}"
    print("Membaca cadangan dari GitHub:", url)
    return pd.read_csv(url, dtype=str, keep_default_na=False)

In [5]:
def transform_sales(df):
    df = df.copy()
    # 1) hapus baris duplikat persis
    df = df.drop_duplicates()
    # 2) tanggal: dua format berbeda -> parse masing-masing lalu gabungkan
    iso = pd.to_datetime(df["transaction_date"], format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(df["transaction_date"], format="%d/%m/%Y", errors="coerce")
    df["transaction_date"] = iso.fillna(dmy).dt.date
    # 3) standarisasi nama kota
    df["city"] = df["city"].str.strip().str.title()
    # 4) quantity kosong -> 1 (asumsi minimal pembelian 1 unit)
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(1).astype(int)
    # 5) harga negatif -> nilai absolut (kesalahan input tanda)
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce").abs()
    # kolom turunan
    df["total_amount"] = df["quantity"] * df["unit_price"]
    return df.reset_index(drop=True)

In [6]:
df_raw = extract_sales(API_URL)
df_clean = transform_sales(df_raw)
print(len(df_clean), 'baris siap dimuat')

Halaman 1: 100 baris
Halaman 2: 100 baris
Halaman 3: 100 baris
Halaman 4: 4 baris
300 baris siap dimuat


## Tugas
1. Muat `df_clean` ke tabel **`sales_clean`** di schema `SCHEMA` (gunakan `if_exists="replace"`, `index=False`).
2. Pastikan kolom `transaction_date` bertipe **DATE** di database (petunjuk: parameter `dtype` dan `from sqlalchemy import Date`).
3. Hitung jumlah baris di database dan simpan ke variabel `n_db`.

In [7]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            current_database() AS database_name,
            current_schema() AS schema_name,
            current_user AS username
    """))

    print(result.fetchone())

    schemas = conn.execute(text("""
        SELECT schema_name
        FROM information_schema.schemata
    """))

    print("Schema tersedia:", schemas.fetchall())

('workshop', 'peserta25', 'peserta25')
Schema tersedia: [('pg_catalog',), ('information_schema',), ('ref',), ('public',), ('peserta25',)]


In [10]:
from sqlalchemy import Date, text

# TODO 1 & 2: Simpan df_clean ke database
df_clean.to_sql(
    name="sales_clean",
    con=engine,
    schema="peserta25",
    if_exists="replace",
    index=False,
    dtype={
        "transaction_date": Date()
    }
)

# TODO 3: Hitung jumlah baris di database
with engine.connect() as conn:
    n_db = conn.execute(
        text("SELECT COUNT(*) FROM peserta25.sales_clean")
    ).scalar()

print("Jumlah baris di database:", n_db)

Jumlah baris di database: 300


## Cek Jawaban

In [11]:
assert n_db == 300, f"n_db = {n_db}, seharusnya 300"
tipe = pd.read_sql(text(f"""SELECT data_type FROM information_schema.columns
    WHERE table_schema='{SCHEMA}' AND table_name='sales_clean' AND column_name='transaction_date'"""), engine)
assert len(tipe) and tipe.iloc[0, 0] == "date", "transaction_date belum bertipe DATE"
print("✅ Exercise 3 benar! Tabel", f"{SCHEMA}.sales_clean", "berisi", n_db, "baris.")

✅ Exercise 3 benar! Tabel peserta25.sales_clean berisi 300 baris.


## Tantangan tambahan — Idempotensi
Jalankan sel Load dua kali dengan `if_exists="append"`. Berapa jumlah barisnya? Mengapa ini berbahaya di pipeline harian, dan bagaimana mencegahnya (misalnya hapus data tanggal yang sama sebelum insert, atau *upsert* dengan primary key)?

In [ ]:
# eksperimen